# 🧬 Splice Acceptor Site Prediction with Deep Learning
## A Comparative Analysis of CNN Architectures on Human Genomic Sequences

---

> **Author:** Daniela Sánchez Aristizábal — Biologist, Computational Biology  
> **Goal:** Apply convolutional neural networks to the biologically critical task of splice site recognition, and interpret model performance in genomic terms.

---

## 1. Biological Background: Why Splicing Matters

In eukaryotic organisms, most protein-coding genes are interrupted by **introns** — non-coding sequences that must be precisely removed during mRNA processing. This process, called **pre-mRNA splicing**, is carried out by a large molecular machine called the **spliceosome**.

The spliceosome recognizes splice sites through short, conserved sequence elements:

```
                   ← intron →
5'—EXON—[GU.....branch point.....polypyrimidine tract.....AG]—EXON—3'
          ↑                                                    ↑
       Donor site                                        Acceptor site
       (5' splice site)                                 (3' splice site)
```

### The Acceptor Splice Site

The **3' splice site** (acceptor) is defined by three elements, all located at the 3' end of the intron:

| Element | Location | Sequence | Function |
|---------|----------|----------|---------|
| Branch point | ~15–40 nt upstream of AG | YNYURAY | Binding site for U2 snRNP; forms the lariat intermediate |
| Polypyrimidine tract | Between branch point and AG | Poly-C/U | Binding site for U2AF65; stabilizes spliceosome assembly |
| AG dinucleotide | Last 2 nt of intron | AG | Conserved consensus; required for 3' cleavage |

### The Prediction Challenge

The human genome contains millions of AG dinucleotides, but only a small fraction are functional splice sites. The question is: **what features in the surrounding sequence context distinguish a true acceptor site from a decoy?**

This is the problem we address with deep learning.

## 2. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

from utils.function import (
    fasta_to_onehot,
    Spliceator, SpliceFinder, DeepSplicer,
    training_process, plot_all_results
)

# Plot styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully')

## 3. Dataset: Description and Loading

### 3.1 Dataset Design

The dataset consists of human genomic sequences in FASTA format:

- **Positive sequences (label = 1):** True acceptor splice sites confirmed in the human genome
- **Negative sequences (label = 0):** Genomic sequences containing the AG dinucleotide but *not* at functional splice sites

Each sequence has a fixed length of **602 nucleotides**, centered on the AG dinucleotide at positions **301–302**.

| Parameter | Value |
|-----------|-------|
| Positive sequences | 3,200 |
| Negative sequences | 3,200 |
| Total sequences | 6,400 |
| Sequence length | 602 nt |
| AG position | 301–302 (center) |

### 3.2 Critical Design Choice

> Both positive AND negative sequences contain the **AG dinucleotide at the same position**. This prevents the model from simply detecting the AG motif — it must learn the biological context that makes some AGs functional splice sites and others not.

This mirrors the real-world challenge faced by the spliceosome: not all AGs are equal.

In [ ]:
# Load and encode sequences
df_data = fasta_to_onehot(
    'data/half_acceptor_test_positive.fasta',
    'data/half_acceptor_test_negative.fasta'
)

print(f'Dataset shape: {df_data.shape}')
print(f'\nClass distribution:')
print(df_data['label'].value_counts().rename({1: 'Positive (splice site)', 0: 'Negative (decoy)'}))
print(f'\nSequence length: {len(df_data["sequence"][0])} nucleotides')
print(f'Encoding matrix shape: {df_data["encoding"][0].shape}')

df_data.head(3)

## 4. Exploratory Data Analysis

Before training, we explore the sequences to understand what biological features might be informative for classification.

### 4.1 Nucleotide Composition: Positive vs. Negative

One hallmark of acceptor splice sites is the **polypyrimidine tract** — a stretch of cytosine (C) and uridine/thymine (T) residues immediately upstream of the AG. We would expect positive sequences to have higher C+T content in the intronic region (positions 1–300) compared to negative sequences.

In [ ]:
def get_composition(sequences):
    """Calculate mean nucleotide frequency at each position."""
    nuc_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    n = len(sequences[0])
    counts = {nuc: np.zeros(n) for nuc in 'ACGT'}
    for seq in sequences:
        for i, nt in enumerate(seq):
            if nt in nuc_map:
                counts[nt][i] += 1
    total = len(sequences)
    return {k: v / total for k, v in counts.items()}

pos_seqs = df_data[df_data['label'] == 1]['sequence'].tolist()
neg_seqs = df_data[df_data['label'] == 0]['sequence'].tolist()

pos_comp = get_composition(pos_seqs)
neg_comp = get_composition(neg_seqs)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
colors = {'A': '#2ecc71', 'C': '#3498db', 'G': '#e74c3c', 'T': '#f39c12'}
positions = np.arange(602)

for ax, comp, title, label in zip(
    axes,
    [pos_comp, neg_comp],
    ['Positive sequences (true acceptor sites)', 'Negative sequences (AG decoys)'],
    ['Positive', 'Negative']
):
    for nuc in 'ACGT':
        ax.plot(positions, comp[nuc], alpha=0.8, linewidth=1, label=nuc, color=colors[nuc])
    ax.axvline(x=300, color='black', linestyle='--', linewidth=1.5, label='AG position (301–302)')
    ax.axvspan(250, 300, alpha=0.07, color='purple', label='Polypyrimidine tract region')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Nucleotide frequency')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(0, 0.6)

axes[-1].set_xlabel('Position along sequence (nt)', fontsize=11)
plt.suptitle('Positional Nucleotide Composition: Positive vs. Negative Sequences',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n🔬 Biological interpretation:')
print('Look for elevated C+T content in the intronic region (positions ~260–300) in positive')
print('sequences — this would indicate the polypyrimidine tract characteristic of acceptor sites.')

### 4.2 Polypyrimidine Tract Content

The **polypyrimidine (Py) tract** is one of the most evolutionarily conserved features of the 3' splice site. It is recognized by the splicing factor U2AF65. We calculate the mean C+T content in the region directly upstream of the AG (positions 251–300) for each class.

In [ ]:
def polypyrimidine_score(sequence, start=250, end=300):
    """Calculate C+T fraction in the polypyrimidine tract region."""
    region = sequence[start:end]
    py_count = region.count('C') + region.count('T')
    return py_count / len(region) if len(region) > 0 else 0

df_data['py_score'] = df_data['sequence'].apply(polypyrimidine_score)

fig, ax = plt.subplots(figsize=(8, 5))

for label, color, name in [(1, '#27ae60', 'Positive (splice site)'), (0, '#e74c3c', 'Negative (decoy)')]:
    scores = df_data[df_data['label'] == label]['py_score']
    ax.hist(scores, bins=30, alpha=0.6, color=color, label=f'{name} (n={len(scores)})', edgecolor='white')

ax.set_xlabel('Polypyrimidine Tract Score (C+T fraction, positions 251–300)', fontsize=11)
ax.set_ylabel('Number of sequences', fontsize=11)
ax.set_title('Polypyrimidine Content Upstream of the AG Dinucleotide', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

pos_mean = df_data[df_data['label'] == 1]['py_score'].mean()
neg_mean = df_data[df_data['label'] == 0]['py_score'].mean()

print(f'\nMean polypyrimidine score — Positive: {pos_mean:.3f} | Negative: {neg_mean:.3f}')
print(f'Difference: {pos_mean - neg_mean:+.3f}')
print('\n🔬 Biological interpretation:')
print('A higher C+T content in positive sequences would confirm that the polypyrimidine')
print('tract is a biologically meaningful feature — one that CNN filters may learn to detect.')

## 5. Data Encoding: From Sequences to Matrices

### 5.1 One-Hot Encoding

Convolutional neural networks operate on numerical tensors, not text. We use **one-hot encoding** to represent each nucleotide as a binary vector:

```
A → [1, 0, 0, 0, 0]
C → [0, 1, 0, 0, 0]
G → [0, 0, 1, 0, 0]
T → [0, 0, 0, 1, 0]
N → [0, 0, 0, 0, 0]  ← ambiguous base
```

A sequence of 602 nucleotides becomes a **matrix of shape (602, 5)**.

### 5.2 Why One-Hot?

- **No arbitrary ordering:** numeric encodings (A=1, C=2, G=3, T=4) would imply false relationships
- **Preserves positional information:** unlike k-mer or bag-of-words approaches
- **CNN-compatible:** convolutional filters scan the matrix like sequence motif scanners

> 💡 **Insight:** A 1D convolutional filter of size *k* applied to a one-hot matrix is mathematically equivalent to a **position-specific scoring matrix (PSSM)** — the classic tool for motif detection in bioinformatics.

In [ ]:
# Visualize one-hot encoding of an example sequence
example_seq = df_data.iloc[0]
example_matrix = np.array(example_seq['encoding'], dtype=float)

# Show the region around the AG site (positions 295–310)
window = example_matrix[295:310, :]
region_seq = example_seq['sequence'][295:310]

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(window.T, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_yticks(range(5))
ax.set_yticklabels(['A', 'C', 'G', 'T', 'N'], fontsize=12)
ax.set_xticks(range(15))
ax.set_xticklabels(list(region_seq), fontsize=11, fontfamily='monospace')
ax.set_xlabel('Position (region around AG site: 295–310)', fontsize=11)
ax.set_title('One-Hot Encoding Matrix — Example Sequence Around the AG Dinucleotide',
             fontsize=11, fontweight='bold')
# Highlight AG position
ax.axvspan(4.5, 6.5, alpha=0.2, color='red', label='AG dinucleotide (pos 301–302)')
ax.legend(fontsize=9)
plt.colorbar(im, ax=ax, label='Nucleotide presence')
plt.tight_layout()
plt.show()

print(f'\nSequence around AG: {region_seq}')
print(f'Full encoding shape: {example_matrix.shape}')
print(f'Label: {"Positive (splice site)" if example_seq["label"] == 1 else "Negative (decoy)"}')

## 6. CNN Architectures for Splice Site Prediction

We implement three architectures inspired by the literature. Each applies convolutional filters to the one-hot matrix — essentially learning sequence motifs from data, without any prior biological knowledge encoded.

### Conceptual analogy

| Traditional bioinformatics | Deep learning equivalent |
|---------------------------|-------------------------|
| Hand-crafted PSSM | Learned convolutional filter |
| Fixed motif window | Learnable kernel size |
| BLAST/alignment | Hierarchical feature extraction |
| Max score across positions | Global max-pooling |

In [ ]:
# Show architecture summaries
input_shape = (602, 5)

arch_data = {
    'Model': ['Spliceator', 'SpliceFinder', 'DeepSplicer'],
    'Conv layers': [3, 1, 1],
    'Kernel sizes': ['7, 6, 6', '9', '11'],
    'Max filters': [64, 50, 16],
    'Regularization': ['Dropout', 'Dropout', 'BatchNorm + L1/L2 + Dropout'],
    'Receptive field': ['Hierarchical (short → long)', 'Local (9 nt)', 'Wide local (11 nt)'],
    'Inspired by': ['Scalzitti et al. 2021', 'Meher et al. 2022', 'Custom design']
}

df_arch = pd.DataFrame(arch_data)
print('Architecture comparison:')
display(df_arch)

# Count parameters
for ModelClass in [Spliceator, SpliceFinder, DeepSplicer]:
    model = ModelClass(input_shape)
    n_params = model.count_params()
    print(f'{ModelClass.__name__}: {n_params:,} trainable parameters')

In [ ]:
# Architecture depth visualization
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

models_info = [
    {
        'name': 'SpliceFinder',
        'layers': [('Input\n(602, 5)', 0.8), ('Conv1D\n50 filters, k=9', 1.4),
                   ('Flatten', 0.6), ('Dense 100', 0.9), ('Output\nSigmoid', 0.5)],
        'color': '#3498db'
    },
    {
        'name': 'Spliceator',
        'layers': [('Input\n(602, 5)', 0.8), ('Conv1D 16\nk=7 + Pool', 1.1),
                   ('Conv1D 32\nk=6 + Pool', 1.3), ('Conv1D 64\nk=6 + Pool', 1.5),
                   ('Dense 100', 0.9), ('Output', 0.5)],
        'color': '#27ae60'
    },
    {
        'name': 'DeepSplicer',
        'layers': [('Input\n(602, 5)', 0.8), ('BatchNorm', 0.6),
                   ('Conv1D 16\nk=11 + Pool', 1.3), ('Dense 32\nL2 reg', 0.9),
                   ('Dense 16\nL2 reg', 0.7), ('Output', 0.5)],
        'color': '#e74c3c'
    }
]

for ax, info in zip(axes, models_info):
    n = len(info['layers'])
    y_positions = np.linspace(0, 1, n)
    
    for i, (label, width) in enumerate(info['layers']):
        rect = plt.Rectangle((0.5 - width/2, y_positions[i] - 0.06),
                              width, 0.1, color=info['color'], alpha=0.7 - i * 0.05,
                              ec='white', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(0.5, y_positions[i], label, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')
        if i < n - 1:
            ax.annotate('', xy=(0.5, y_positions[i+1] - 0.07),
                        xytext=(0.5, y_positions[i] + 0.05),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.05, 1.05)
    ax.axis('off')
    ax.set_title(info['name'], fontsize=13, fontweight='bold', color=info['color'])

plt.suptitle('CNN Architecture Diagrams', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Model Training: K-Fold Cross-Validation

### Why K-Fold?

In genomics, datasets can be small relative to the complexity of the underlying biology. A single train/test split can give misleading results due to the particular sequences that end up in each partition. K-Fold cross-validation:

- Provides a **more reliable estimate** of generalization performance
- Uses all data for both training and validation
- Allows detection of **variance** across different subsets

### Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|----------|
| Folds | 5 | Standard; balances bias-variance |
| Optimizer | Adam | Adaptive learning rate; robust to choice of lr |
| Loss | Binary cross-entropy | Standard for binary classification |
| Batch size | 32 | Balances gradient stability and memory |
| Metric suite | Accuracy, Precision, Recall, AUC, F1 | Comprehensive genomic evaluation |

In [ ]:
cnns = [Spliceator, SpliceFinder, DeepSplicer]

print('Starting training pipeline...')
print('=' * 60)

results = training_process(df_data, n_folds=5, cnns=cnns)

print('\n✅ Training complete.')

## 8. Results and Biological Interpretation

### 8.1 Quantitative Performance Comparison

In [ ]:
df_metrics = results['evaluation_metrics'].copy()
df_metrics = df_metrics.set_index('cnn')

# Style the table
styled = df_metrics.style\
    .format({
        'loss': '{:.4f}',
        'accuracy': '{:.4f}',
        'precision': '{:.4f}',
        'recall': '{:.4f}',
        'auc': '{:.4f}',
        'f1': '{:.4f}'
    })\
    .highlight_max(subset=['accuracy', 'precision', 'recall', 'auc', 'f1'],
                   props='background-color: #d4edda; color: #155724; font-weight: bold')\
    .highlight_min(subset=['loss'],
                   props='background-color: #d4edda; color: #155724; font-weight: bold')\
    .set_caption('Table 1. Mean performance metrics across cross-validation folds (highlighted = best).')

display(styled)

# Print biological interpretation
best_auc = df_metrics['auc'].idxmax()
best_recall = df_metrics['recall'].idxmax()
best_f1 = df_metrics['f1'].idxmax()

print(f'\n📊 Summary:')
print(f'   Best AUC:    {best_auc} ({df_metrics.loc[best_auc, "auc"]:.4f})')
print(f'   Best Recall: {best_recall} ({df_metrics.loc[best_recall, "recall"]:.4f})')
print(f'   Best F1:     {best_f1} ({df_metrics.loc[best_f1, "f1"]:.4f})')

### 8.2 Biological Meaning of Each Metric

| Metric | Formula | Biological meaning in splice site context |
|--------|---------|-------------------------------------------|
| **Accuracy** | (TP+TN) / Total | Overall classification rate |
| **Precision** | TP / (TP+FP) | Of all predicted splice sites, how many are real? High precision = fewer false alarms |
| **Recall (Sensitivity)** | TP / (TP+FN) | Of all true splice sites, how many were found? High recall = fewer missed splice sites |
| **AUC-ROC** | Area under ROC | Discrimination ability across all decision thresholds |
| **F1-Score** | 2 × (P×R)/(P+R) | Harmonic mean — useful when precision-recall trade-off matters |

> 🧬 **Key insight:** In a clinical variant annotation context (e.g., identifying pathogenic splice variants), **recall is critical**. A missed splice site mutation (false negative) could mean failing to flag a disease-causing variant. However, too many false positives (low precision) would make the tool impractical for variant interpretation.

In [ ]:
# Visualize metrics comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'auc', 'f1']
model_names = df_metrics.index.tolist()
colors_models = ['#27ae60', '#3498db', '#e74c3c']

x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))

for i, (model, color) in enumerate(zip(model_names, colors_models)):
    values = [df_metrics.loc[model, m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=model, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels([m.upper() for m in metrics_to_plot], fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title('Performance Metrics Comparison Across CNN Architectures\n(averaged across cross-validation folds)',
             fontsize=12, fontweight='bold')
ax.legend(title='Model', fontsize=10)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random baseline')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 8.3 Training Dynamics: Loss Curves

Loss curves reveal how each model learns over epochs and whether it overfits to training data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, (model_name, color) in zip(axes, zip(model_names, colors_models)):
    histories = results['training_curves'][model_name]
    all_train = np.array([h['loss'] for h in histories])
    all_val = np.array([h['val_loss'] for h in histories])
    
    mean_train = all_train.mean(axis=0)
    mean_val = all_val.mean(axis=0)
    std_train = all_train.std(axis=0)
    std_val = all_val.std(axis=0)
    
    epochs = np.arange(1, len(mean_train) + 1)
    
    ax.plot(epochs, mean_train, '-o', color=color, label='Training loss', linewidth=2)
    ax.plot(epochs, mean_val, '--s', color=color, alpha=0.6, label='Validation loss', linewidth=2)
    ax.fill_between(epochs, mean_train - std_train, mean_train + std_train, alpha=0.15, color=color)
    ax.fill_between(epochs, mean_val - std_val, mean_val + std_val, alpha=0.08, color=color)
    
    ax.set_title(model_name, fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Epoch', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Binary Cross-Entropy Loss', fontsize=11)
plt.suptitle('Training vs. Validation Loss (mean ± std across folds)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔬 Interpretation guide:')
print('• Large gap (train << val): overfitting — model memorizes training sequences')
print('• Both curves converge: good generalization — model learned transferable features')
print('• Both curves stay high: underfitting — model may need more capacity or training')

### 8.4 ROC Curve Analysis

The **Receiver Operating Characteristic (ROC)** curve plots sensitivity (true positive rate) against 1-specificity (false positive rate) across all decision thresholds. The **AUC** (Area Under the Curve) summarizes discrimination ability in a single value.

| AUC range | Interpretation |
|-----------|---------------|
| 1.0 | Perfect classifier |
| 0.9–1.0 | Excellent |
| 0.8–0.9 | Good |
| 0.7–0.8 | Fair |
| 0.5–0.7 | Poor |
| 0.5 | Random (no discriminative ability) |

In [ ]:
from sklearn.metrics import auc as compute_auc

fig, ax = plt.subplots(figsize=(8, 7))

for model_name, color in zip(model_names, colors_models):
    rocs = results['roc_curves'][model_name]
    mean_fpr = np.linspace(0, 1, 100)
    interp_tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in rocs]
    mean_tpr = np.mean(interp_tprs, axis=0)
    std_tpr = np.std(interp_tprs, axis=0)
    model_auc = compute_auc(mean_fpr, mean_tpr)
    
    ax.plot(mean_fpr, mean_tpr, color=color, linewidth=2.5,
            label=f'{model_name}  (AUC = {model_auc:.3f})')
    ax.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr, alpha=0.1, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random baseline (AUC = 0.500)')
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('Mean ROC Curves Across Cross-Validation Folds', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\n🔬 Biological interpretation:')
print('Models above the diagonal line (random baseline) have learned some discriminative')
print('information about what makes a genomic AG a true splice site. The farther toward')
print('the top-left corner, the better the model balances sensitivity and specificity.')

### 8.5 Confusion Matrix Analysis

Confusion matrices reveal the *types* of errors each model makes — a biologically important distinction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (model_name, color) in zip(axes, zip(model_names, colors_models)):
    cm = np.sum(results['confusion_matrices'][model_name], axis=0)
    total = cm.sum()
    cm_pct = cm / total * 100
    
    # Custom colormap centered on this model's color
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues', linewidths=0.5,
                xticklabels=['Pred: Negative\n(Decoy)', 'Pred: Positive\n(Splice site)'],
                yticklabels=['True: Negative\n(Decoy)', 'True: Positive\n(Splice site)'],
                cbar=False)
    
    # Add percentage annotations
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.7, f'({cm_pct[i,j]:.1f}%)',
                    ha='center', fontsize=9, color='gray')
    
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{model_name}\n'
                 f'TP={tp}  FP={fp}  FN={fn}  TN={tn}',
                 fontsize=10, fontweight='bold', color=color)

plt.suptitle('Aggregated Confusion Matrices (all folds combined)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔬 Error interpretation:')
print('• False Positives (FP): Decoy AG sequences wrongly classified as splice sites')
print('  → The model may be detecting the AG motif or partial polypyrimidine patterns')
print('  → In practice: would flag benign variants as potentially splice-disrupting')
print()
print('• False Negatives (FN): True splice sites missed by the model')
print('  → The model failed to recognize the sequence context of these functional AGs')
print('  → In practice: would miss pathogenic splice site variants in clinical analysis')

## 9. Biological Discussion

### 9.1 What Did Each Model Learn?

Despite lacking explicit biological knowledge, CNN models learn sequence representations through their convolutional filters. We can reason about what features each architecture is *capable* of capturing based on its design:

**SpliceFinder (shallow, k=9):**
- A single filter of 9 nt can detect the **polypyrimidine–AG junction** region
- Cannot model long-range elements like the branch point (15–40 nt upstream)
- Fast to train; interpretable but limited in biological scope

**Spliceator (deep, k=7/6/6, hierarchical):**
- First layer: short motifs (7 nt) → potential polypyrimidine-like patterns
- Second/third layers: wider contexts (effective receptive field expands with each pool+conv)
- Best architecture for **hierarchical splicing signals**

**DeepSplicer (wide kernel, regularized):**
- Single conv with k=11: captures slightly wider context than SpliceFinder
- Heavy regularization prevents overfitting to dataset-specific patterns
- May be more robust when applied to novel sequence types

### 9.2 Limitations and Considerations

1. **This is a simplified benchmark** — real-world splice site prediction involves:
   - Much larger datasets (millions of sites across species)
   - Alternative splicing events (cassette exons, intron retention, etc.)
   - Species-specific differences in splicing signals

2. **The 602 nt window** captures most of the canonical 3' splice site elements, but misses some distant regulatory elements (e.g., intronic splicing silencers hundreds of nt away)

3. **One-hot encoding** does not capture:
   - DNA secondary structure
   - Epigenetic modifications
   - Conservation scores across species (could be added as additional channels)

### 9.3 Key Insight: Architecture Matters Biologically

The better-performing models in this benchmark are those that can integrate sequence information across **multiple scales simultaneously** — mirroring the biological reality that splice site recognition involves:

- **Short-range signals:** The AG dinucleotide itself (~2 nt)
- **Medium-range signals:** The polypyrimidine tract (~15–40 nt)
- **Long-range signals:** The branch point sequence (~50–100 nt upstream)

Deep architectures with hierarchical convolutions are naturally suited to learning this multi-scale representation.

## 10. Save All Visualizations

In [ ]:
model_names_list = [cnn.__name__ for cnn in cnns]

plot_all_results(results, model_names_list, save_path='results')

print('📁 Results saved to results/ directory:')
print('   results/curves/    → Loss curves')
print('   results/roc/       → ROC curves')
print('   results/metrics/   → Bar plots')
print('   results/confusion/ → Confusion matrices')

## 11. Conclusions

This project demonstrates that **convolutional neural networks can learn to distinguish functional splice acceptor sites from AG-containing decoy sequences**, even when the canonical AG dinucleotide is present in all sequences.

### Summary of findings:

| Finding | Biological implication |
|---------|------------------------|
| Models perform above random baseline | Meaningful sequence context exists beyond the AG motif |
| AUC variation across architectures | Architectural depth and receptive field matter for genomic tasks |
| Recall vs. precision trade-off visible | Clinical deployment would require threshold optimization |
| Spliceator shows hierarchical advantage | Multi-scale splicing signals benefit from deep CNNs |

### What this means for genomics:

Deep learning models can serve as **data-driven motif discoverers** for genomic classification tasks. Unlike traditional PSSM-based methods, they:
- Require no prior biological assumptions about what features matter
- Can learn complex, non-linear combinations of sequence features
- Scale efficiently to genome-wide datasets

As genomic sequencing becomes standard in clinical settings, tools like these could contribute to **variant effect prediction** and **precision medicine** workflows.

---

**Next steps:** Extend to donor sites, train on the full dataset, add attention mechanisms for interpretability, and benchmark against established tools (MaxEntScan, SpliceAI).

---
*Notebook created by Daniela Sánchez Aristizábal — Computational Biology Portfolio*